In [1]:
import pandas as pd
import numpy as np
import importlib
import time


from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

from xgboost import XGBRegressor as XGB
from sklearn.ensemble import RandomForestRegressor as RF

from sklearn.tree import DecisionTreeRegressor as DTR

from sklearn.neural_network import MLPRegressor

from sklearn.linear_model import LinearRegression as LR

from sklearn.metrics import mean_squared_error

from sklearn.pipeline import Pipeline



import sys
sys.path.append("../")

import src.features

importlib.reload(src.features)
from src.features import generate_features as gen
from src.features import get_dataset, get_periods








from src.evaluate_prediction import prediction_df, evaluate_prediction, evaluate_Models
importlib.reload(src.evaluate_prediction)


ModuleNotFoundError: No module named 'sklearn'

In [35]:
Periods= get_periods("1970-01-31","2020-12-31")

In [36]:
# The circus of models

DTRmodel=DTR(max_leaf_nodes=20)
RFmodel=RF(n_estimators=100)


XGBmodel1=XGB(
    n_estimators=100,
    learning_rate=0.03,
    max_depth=2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10,
    reg_alpha=1,
    random_state=42
)

XGBmodel2=XGB(
    n_estimators=300,
    learning_rate=0.01,
    max_depth=2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10,
    reg_alpha=1,
    random_state=42
)


NN1 = MLPRegressor(
    hidden_layer_sizes=(16),
    activation="relu",
    max_iter=100,
    random_state=42
)
NN2 = MLPRegressor(
    hidden_layer_sizes=(64),
    activation="relu",
    max_iter=200,
    random_state=42
)
NN3 = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    max_iter=200,
    random_state=42
)
NN4 = MLPRegressor(
    hidden_layer_sizes=(64, 32,16),
    activation="relu",
    max_iter=200,
    random_state=42
)



Model= {
    'DTR model': DTRmodel,
    'RF model': RFmodel,
    'XGBmodel1': XGBmodel1,
    'XGBmodel2': XGBmodel2,
    'NN1': NN1,
    'NN2': NN2,
    'NN3': NN3,
    'NN4': NN4,

}


In [37]:

k=0
for i in Periods.index:
    Dataset= get_dataset(Periods.iloc[i])
    Z_train=Dataset["Z_train"]
    y_train=Dataset["y_train"]
    Z_valid=Dataset["Z_valid"]
    y_valid=Dataset["y_valid"]
    for names, model in Model.items():
        model.fit(Z_train,y_train)
    info= pd.DataFrame({
        'start': [y_valid.index[0]],
        'end':[y_valid.index[-1]]
    })
    display(info)
    display(y_valid.columns)
    eval= evaluate_Models(Z_train,y_train,Z_valid,y_valid,Model)
    eval= eval.sort_values("R2_valid", ascending=False)
    display(eval)
    if k==0:
        break
    k+=1



,start,end
0,1990-01-31,1994-12-31


Index(['AEP', 'BA', 'CAT', 'CVX', 'DIS', 'ED', 'IBM', 'IP', 'JNJ', 'KO', 'MCD',
       'MMM', 'MO', 'MRK', 'PG', 'XOM'],
      dtype='str')

,name,mqe_train,R2_train,mqe_valid,R2_valid
3,XGBmodel1,0.055017,0.216049,0.050046,0.024914
4,XGBmodel2,0.054992,0.216742,0.050061,0.024319
0,zero_guess,0.062137,0.000000,0.050682,0.000000
2,RF model,0.022440,0.869577,0.050940,-0.010207
1,DTR model,0.046499,0.440005,0.059315,-0.369732
8,NN4,0.066306,-0.138687,0.181306,-11.797492
7,NN3,0.073177,-0.386906,0.290564,-31.868742
6,NN2,0.097892,-1.481933,0.440465,-74.530760
5,NN1,0.087088,-0.964341,0.453331,-79.007660
